In [1]:
import pandas as pd

train_df = pd.read_csv(
    "data/processed/train_20k.csv"
)
print("Available columns:")
print(train_df.columns.tolist())

print("\nDomain distribution:")
print(train_df["domain"].value_counts().head(20))

print("\nMissing domain values:")
print(train_df["domain"].isna().sum())

Available columns:
['id', 'clean_title', 'image_url', 'domain', 'subreddit', 'created_utc', '2_way_label']

Domain distribution:
domain
i.redd.it              6531
i.imgur.com            2797
imgur.com               969
i.reddituploads.com     123
reuters.com             115
theonion.com            115
archive.is              110
theguardian.com          87
bbc.com                  83
bbc.co.uk                77
washingtonpost.com       75
clickhole.com            75
cnn.com                  74
web.archive.org          70
nytimes.com              70
cbc.ca                   70
youtube.com              68
independent.co.uk        49
v.redd.it                45
foxnews.com              44
Name: count, dtype: int64

Missing domain values:
4976


In [3]:
# now we have three domain 
# Image hosting
# news website 
# other/unknown

# so we can catgorized this domain in the seperate way

import numpy as np 

train_df = pd.read_csv(
    "data/processed/train_20k.csv"
)

# catogrizes the domain type 

def categorize_domain(domain):
    if pd.isna(domain):
        return "Unknown"

    domain = str(domain).lower()

    image_domains = [
        "i.redd.it",
        "i.imgur.com",
        "imgur.com",
        "i.reddituploads.com",
        "v.redd.it"
    ]
    if domain in image_domains:
        return "Image Hosting"

    news_domain = [
        "reuters.com",
        "theguardian.com",
        "bbc.com",
        "bbc.co.uk",
        "washingtonpost.com",
        "cnn.com",
        "nytimes.com",
        "cbc.ca",
        "independent.co.uk",
        "foxnews.com"
    ]

    if domain in news_domain:
        return "News Publisher"

    return "other"

train_df["domain_category"] = train_df["domain"].apply(
    categorize_domain
)

print(train_df["domain_category"].value_counts())

print("\nSample results:")
print(train_df[["domain" , "domain_category"]].head(20))

domain_category
Image Hosting     10465
Unknown            4976
other              3815
News Publisher      744
Name: count, dtype: int64

Sample results:
               domain domain_category
0           i.redd.it   Image Hosting
1         i.imgur.com   Image Hosting
2                 NaN         Unknown
3           i.redd.it   Image Hosting
4          nypost.com           other
5         reuters.com  News Publisher
6           i.redd.it   Image Hosting
7           i.redd.it   Image Hosting
8           i.redd.it   Image Hosting
9       instagram.com           other
10                NaN         Unknown
11                NaN         Unknown
12   screen.yahoo.com           other
13          i.redd.it   Image Hosting
14  thedailybeast.com           other
15                NaN         Unknown
16            nfb.org           other
17       sbnation.com           other
18          i.redd.it   Image Hosting
19                NaN         Unknown


In [ ]:
# create the cource creatability 

#Create numerical source features

train_df["is_missing_domain"] = train_df["domain"].isna().astype(int)

train_df["is_image_host"] = (
    train_df["domain_category"] == "Image Hosting"
).astype(int)

train_df["is_known_publisher"] = (
    train_df["domain_category"] == "News Publisher"
).astype(int)

# Domain freqency (calculate by training data only)

domain_counts = train_df["domain"].value_counts()

train_df["domain_frequency"] = (
    train_df["domain"].map(domain_counts).fillna(0)
)

print("Source credibility features created!")

print(train_df[
    ["domain", "domain_category",
     "is_missing_domain", "is_image_host",
     "is_known_publisher", "domain_frequency"]
].head(15))

print("\nFeature summary:")
print(train_df[
    ["is_missing_domain", "is_image_host",
     "is_known_publisher", "domain_frequency"]
].describe())


# Prepare source feature for machine learning

In [5]:

import numpy as np
import pandas as pd

# Reduce skew in domain frequency
train_df["log_domain_frequency"] = np.log1p(
    train_df["domain_frequency"]
)

# One-hot encode domain categories
domain_encoded = pd.get_dummies(
    train_df["domain_category"],
    prefix="domain",
    dtype=int
)

# Select numerical source features
source_features = pd.concat(
    [
        domain_encoded,
        train_df[
            [
                "is_missing_domain",
                "is_image_host",
                "is_known_publisher",
                "log_domain_frequency"
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

print("Source feature shape:", source_features.shape)
print("\nSource feature columns:")
print(source_features.columns.tolist())

print("\nFirst 5 rows:")
print(source_features.head())


Source feature shape: (20000, 8)

Source feature columns:
['domain_Image Hosting', 'domain_News Publisher', 'domain_Unknown', 'domain_other', 'is_missing_domain', 'is_image_host', 'is_known_publisher', 'log_domain_frequency']

First 5 rows:
   domain_Image Hosting  domain_News Publisher  domain_Unknown  domain_other  \
0                     1                      0               0             0   
1                     1                      0               0             0   
2                     0                      0               1             0   
3                     1                      0               0             0   
4                     0                      0               0             1   

   is_missing_domain  is_image_host  is_known_publisher  log_domain_frequency  
0                  0              1                   0              8.784468  
1                  0              1                   0              7.936660  
2                  1              0   

In [7]:
# check whether the dataset match 


# Load the image dataset used in multimodal fusion
image_df = pd.read_csv("data/processed/image_train_1889.csv")

# Check matching IDs
image_ids = set(image_df["id"].astype(str))
source_ids = set(train_df["id"].astype(str))

matching_ids = image_ids.intersection(source_ids)

print("Image dataset records:", len(image_df))
print("Source dataset records:", len(train_df))
print("Matching IDs:", len(matching_ids))
print(
    "Matching percentage:",
    round(len(matching_ids) / len(image_df) * 100, 2),
    "%"
)


Image dataset records: 1889
Source dataset records: 20000
Matching IDs: 1889
Matching percentage: 100.0 %


# Align source features with image records

In [8]:

# Match source records with image dataset IDs
source_subset = train_df[
    train_df["id"].astype(str).isin(image_ids)
].copy()

# Keep exactly the same ID order as image_df
source_subset["id"] = source_subset["id"].astype(str)

source_subset = (
    source_subset
    .set_index("id")
    .loc[image_df["id"].astype(str)]
    .reset_index()
)

# Create aligned source features
domain_encoded_subset = pd.get_dummies(
    source_subset["domain_category"],
    prefix="domain",
    dtype=int
)

source_features_subset = pd.concat(
    [
        domain_encoded_subset.reset_index(drop=True),
        source_subset[
            [
                "is_missing_domain",
                "is_image_host",
                "is_known_publisher",
                "log_domain_frequency"
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

print("Aligned source feature shape:", source_features_subset.shape)

print(
    "IDs aligned:",
    np.array_equal(
        source_subset["id"].to_numpy(),
        image_df["id"].astype(str).to_numpy()
    )
)

print("\nFirst 5 source feature rows:")
print(source_features_subset.head())


Aligned source feature shape: (1889, 8)
IDs aligned: True

First 5 source feature rows:
   domain_Image Hosting  domain_News Publisher  domain_Unknown  domain_other  \
0                     1                      0               0             0   
1                     1                      0               0             0   
2                     0                      0               1             0   
3                     1                      0               0             0   
4                     0                      0               0             1   

   is_missing_domain  is_image_host  is_known_publisher  log_domain_frequency  
0                  0              1                   0              8.784468  
1                  0              1                   0              7.936660  
2                  1              0                   0              0.000000  
3                  0              1                   0              8.784468  
4                  0           

In [21]:
# just reload the file
import numpy as np

text_train_features = np.load(
    "../features/distilbert_train_features.npy"
)

text_val_features = np.load(
    "../features/distilbert_val_features.npy"
)

print("Training text features:", text_train_features.shape)
print("Validation text features:", text_val_features.shape)

Training text features: (1511, 768)
Validation text features: (378, 768)


In [23]:
# combine all three feature 
from sklearn.model_selection import train_test_split
import numpy as np

# Recreate the same train-validation split
indices = np.arange(len(image_df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=image_df["2_way_label"]
)

# Split source features using the correct variable
source_train_features = source_features_subset.iloc[
    train_idx
].to_numpy(dtype=np.float32)

source_val_features = source_features_subset.iloc[
    val_idx
].to_numpy(dtype=np.float32)

# Combine text + image + source features
X_train_full = np.concatenate(
    [text_train_features, image_features, source_train_features],
    axis=1
)

X_val_full = np.concatenate(
    [text_val_features, val_features, source_val_features],
    axis=1
)

# Labels
y_train_full = image_df.iloc[train_idx]["2_way_label"].to_numpy()
y_val_full = image_df.iloc[val_idx]["2_way_label"].to_numpy()

print("Training feature shape:", X_train_full.shape)
print("Validation feature shape:", X_val_full.shape)
print("Training labels:", y_train_full.shape)
print("Validation labels:", y_val_full.shape)


Training feature shape: (1511, 1288)
Validation feature shape: (378, 1288)
Training labels: (1511,)
Validation labels: (378,)


# train the text +  image + source model

In [24]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Create the classifier
source_fusion_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Train the model
source_fusion_model.fit(X_train_full, y_train_full)

print("Text + Image + Source model trained successfully!")

# Make predictions
y_pred_full = source_fusion_model.predict(X_val_full)

# Evaluate performance
accuracy = accuracy_score(y_val_full, y_pred_full)
precision = precision_score(y_val_full, y_pred_full)
recall = recall_score(y_val_full, y_pred_full)
f1 = f1_score(y_val_full, y_pred_full)

print("\nModel Performance:")
print("Accuracy :", round(accuracy * 100, 2), "%")
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\nClassification Report:")
print(classification_report(y_val_full, y_pred_full))

print("Confusion Matrix:")
print(confusion_matrix(y_val_full, y_pred_full))


Text + Image + Source model trained successfully!

Model Performance:
Accuracy : 84.66 %
Precision: 0.8555
Recall   : 0.8177
F1 Score : 0.8362

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.87      0.86       197
           1       0.86      0.82      0.84       181

    accuracy                           0.85       378
   macro avg       0.85      0.85      0.85       378
weighted avg       0.85      0.85      0.85       378

Confusion Matrix:
[[172  25]
 [ 33 148]]


In [25]:
# save current model and results

import joblib
import json 
import os 

os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

# save the model 

joblib.dump(
    source_fusion_model,
     "../models/text_image_source_model.pkl"
)

# Save evaluation results
results = {
    "model": "Text + Image + Source",
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "validation_samples": len(y_val_full)
}

with open("../results/source_fusion_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Source fusion model saved successfully!")
print("Evaluation results saved successfully!")

Source fusion model saved successfully!
Evaluation results saved successfully!
